<a href="https://colab.research.google.com/github/NietoEmmanuel/SIMULACION-I/blob/main/LineadeEspera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SISTEMA DE LINEA DE ESPERA CON UN SERVIDOR



Considere una estación de servicio a la cual los clientes llegan de acuerdo con un proceso Poisson no homogéneo con función de intensidad λ(t), t ≥ 0. Hay un único servidor, y al llegar un cliente pasa a servicio si el servidor está libre en ese momento, o bien se une a la fila de espera si está ocupado. Cuando el servidor termina de dar servicio a un cliente, se ocupa del cliente que ha estado esperando más tiempo (la disciplina "primero en llegar, primero en atender") si hay clientes esperando, o bien, si no los hay, permanece libre hasta la llegada del siguiente cliente. El tiempo que tarda la atención a un cliente es una variable aleatoria independiente de los demás tiempos de servicio y de llegada con distribución de probabilidad G. Además, hay un tiempo fijo T después del cual no se permite otra llegada al sistema, aunque el servidor atiende a todos los que ya están dentro del sistema en el instante T.


In [16]:
import numpy as np
import random as r
from collections import deque

In [17]:
def LineaEspera(T, lamb, mu):

    # Reloj de la simulación y estado del sistema
    t = 0.0
    n = 0
    ultimo_evento = 0.0

    # Total de llegadas y salidas registradas
    num_llegadas = 0
    num_salidas = 0

    # Acumuladores para calcular promedios al final
    area_sistema = 0.0
    area_cola = 0.0
    tiempo_ocupado = 0.0
    suma_W = 0.0
    suma_Wq = 0.0

    # Cola de espera y variables del servidor
    cola = deque()
    llegada_en_servicio = None
    servicio_actual = None

    # Tiempos de la lista de eventos
    U = r.random()
    tA = -(1 / lamb) * np.log(U)
    tD = float("inf")

    while True:

        t_evento = min(tA, tD)

        # Acumular área antes de procesar el evento
        dt = t_evento - ultimo_evento
        area_sistema += n * dt
        area_cola += max(n - 1, 0) * dt

        if n > 0:
            tiempo_ocupado += dt

        ultimo_evento = t_evento

        # CASO 1: llega un cliente antes de que salga alguno y antes de T
        if tA <= tD and tA <= T:

            t = tA

            num_llegadas += 1
            n += 1

            cola.append(t)

            # Generar siguiente llegada
            U = r.random()
            tA = t + (-(1 / lamb) * np.log(U))

            # Si el servidor estaba libre, atender de inmediato
            if n == 1:

                llegada_en_servicio = cola.popleft()

                U = r.random()
                servicio_actual = -(1 / mu) * np.log(U)

                tD = t + servicio_actual

        # CASO 2: sale un cliente antes de que llegue alguno y antes de T
        elif tD < tA and tD <= T:

            t = tD

            n -= 1
            num_salidas += 1

            W = t - llegada_en_servicio
            Wq = W - servicio_actual

            suma_W += W
            suma_Wq += Wq

            # Si quedan clientes, atender al siguiente en cola
            if n > 0:

                llegada_en_servicio = cola.popleft()

                U = r.random()
                servicio_actual = -(1 / mu) * np.log(U)

                tD = t + servicio_actual

            else:

                llegada_en_servicio = None
                servicio_actual = None
                tD = float("inf")

        # CASO 3: se superó T pero aún hay clientes, terminar de atenderlos
        elif min(tA, tD) > T and n > 0:

            t = tD

            n -= 1
            num_salidas += 1

            W = t - llegada_en_servicio
            Wq = W - servicio_actual

            suma_W += W
            suma_Wq += Wq

            if n > 0:

                llegada_en_servicio = cola.popleft()

                U = r.random()
                servicio_actual = -(1 / mu) * np.log(U)

                tD = t + servicio_actual

            else:

                llegada_en_servicio = None
                servicio_actual = None
                tD = float("inf")

        # CASO 4: se superó T y el sistema quedó vacío, terminar
        elif min(tA, tD) > T and n == 0:

            break

    # Resultados de simulación
    rho_sim = round(tiempo_ocupado / T, 6)
    L_sim   = round(area_sistema / T, 6)
    Lq_sim  = round(area_cola / T, 6)
    W_sim   = round(suma_W / num_salidas, 6)
    Wq_sim  = round(suma_Wq / num_salidas, 6)


    # Fórmulas analíticas M/M/1
    rho = round(lamb / mu, 6)
    P0  = round(1 - rho, 6)
    Lq  = round(lamb**2 / (mu * (mu - lamb)), 6)
    Ls  = round(Lq + (lamb / mu), 6)
    Wq  = round(Lq / lamb, 6)
    W   = round(Wq + (1 / mu), 6)


    print("\n========== TEÓRICO ==========")
    print("ρ =", rho)
    print("P0 =", P0)
    print("Lq =", Lq)
    print("Ls =", Ls)
    print("Wq =", Wq)
    print("W =", W)

    print("========== SIMULACIÓN ==========")
    print("Utilización promedio (ρ):", rho_sim)
    print("Clientes promedio en cola (Lq):", Lq_sim)
    print("Clientes promedio en sistema (L):", L_sim)
    print("Tiempo promedio en cola (Wq):", Wq_sim)
    print("Tiempo promedio en sistema (W):", W_sim)



    print("\n========== ERROR (%) ==========")
    print("Error ρ  :", round(abs(rho_sim - rho) / rho * 100, 6), "%")
    print("Error Lq :", round(abs(Lq_sim - Lq) / Lq * 100, 6), "%")
    print("Error Ls :", round(abs(L_sim - Ls) / Ls * 100, 6), "%")
    print("Error Wq :", round(abs(Wq_sim - Wq) / Wq * 100, 6), "%")
    print("Error W  :", round(abs(W_sim - W) / W * 100, 6), "%")
    return




Al final compararemos el resultado simulado con las ecuaciones teoricas y calcularemos el error:

###1.    Factor de utilización
$$
\rho = \frac{\lambda}{\mu}
$$

###2.  Probabilidad de que no haya unidades en el sistema

$$
P_0 = 1 - \frac{\lambda}{\mu}
$$


###3. Número promedio de unidades en cola

$$
L_q = \frac{\lambda^2}{\mu(\mu-\lambda)}
$$

###4.  Número promedio de unidades en el sistema

$$
L_s = L_q + \frac{\lambda}{\mu}
$$

###5.  Tiempo promedio que una unidad pasa en una cola

$$
W_q = \frac{L_q}{\lambda}
$$

###6.  Tiempo promedio que una unidad pasa en el sistema

$$
W_s = W_q + \frac{1}{\mu}
$$

In [18]:
LineaEspera(100000, 4, 6)


========== TEÓRICO ==========
ρ = 0.666667
P0 = 0.333333
Lq = 1.333333
Ls = 2.0
Wq = 0.333333
W = 0.5
========== SIMULACIÓN ==========
Utilización promedio (ρ): 0.66416
Clientes promedio en cola (Lq): 1.319419
Clientes promedio en sistema (L): 1.983579
Tiempo promedio en cola (Wq): 0.331165
Tiempo promedio en sistema (W): 0.497865

========== ERROR (%) ==========
Error ρ  : 0.37605 %
Error Lq : 1.04355 %
Error Ls : 0.82105 %
Error Wq : 0.650401 %
Error W  : 0.427 %
